# Underground Nexus: Agentic Workspace via MCP

Welcome to the **Nexus Workbench**. This notebook demonstrates how to connect to our local Model Context Protocol (MCP) server (`nexus-mcp`) from within JupyterLab to interact with SOC tools (like Wazuh) and AI models programmatically.

In [ ]:
import asyncio
from mcp import ClientSession, StdioServerParameters
from mcp.client.sse import sse_client

# The nexus-mcp server runs locally in the 'soc' namespace and exposes SSE on port 3001.
MCP_SERVER_URL = "http://nexus-mcp.soc.svc.cluster.local:3001/sse"

### 1. Connect to the MCP Server
Let's establish a connection to the server using Server-Sent Events (SSE) and list the available tools.

In [ ]:
async def list_tools():
    async with sse_client(MCP_SERVER_URL) as (read_stream, write_stream):
        async with ClientSession(read_stream, write_stream) as session:
            await session.initialize()
            
            # List available tools
            tools = await session.list_tools()
            print("Available MCP Tools:")
            for tool in tools.tools:
                print(f"- {tool.name}: {tool.description}")
            
            return tools

# Run the async function
tools = await list_tools()

### 2. Execute a Tool
Let's call the `get_wazuh_alerts` tool (assuming it's exposed by the `nexus-mcp` server).

In [ ]:
async def fetch_alerts():
    async with sse_client(MCP_SERVER_URL) as (read_stream, write_stream):
        async with ClientSession(read_stream, write_stream) as session:
            await session.initialize()
            
            try:
                # Example tool call
                result = await session.call_tool("get_wazuh_alerts", arguments={"limit": 5})
                print("Tool Result:")
                print(result)
            except Exception as e:
                print(f"Tool execution failed (or tool doesn't exist yet): {e}")

await fetch_alerts()